In [1]:
from pathlib import Path
import pandas as pd

In [2]:
# Paths
label_path = '/home/ec2-user/Jul2025/labels/' # label folder with all the labels

# Orre et al. 3365 markers
orre_marker_path = Path(label_path+'markers.txt') # path to the orre marker file
orre_mcluster_path = Path(label_path+'markers_mclusters.txt') # path to the orre mcluster file

# Uniprot Go markers
uniprot_marker_path = Path(label_path+'uniprot_go_markers_grouped.txt') # path to the uniprot go marker file

In [3]:
# Load reference label tables
ld_path = Path(label_path) / 'uniprot_go_markers_grouped.txt'
LD = (
    pd.read_csv(ld_path, sep='\t')
      .rename(columns={'Protein': 'Protein_ID'})
      .set_index('Protein_ID')
)

marker_path = Path(label_path) / 'markers.txt'
LD_marker = (
    pd.read_csv(marker_path, sep='\t')
      .rename(columns={'Protein': 'Protein_ID'})
      .set_index('Protein_ID')
)

# Reconcile marker localizations with LD where they disagree
LD_marker_new = LD_marker.copy()
shared_proteins = LD_marker_new.index.intersection(LD.index)

mismatch_mask = LD_marker_new.loc[shared_proteins, 'Localization'] != LD.loc[shared_proteins, 'Localization']
proteins_updated = shared_proteins[mismatch_mask]

LD_marker_new.loc[proteins_updated, 'Localization'] = LD.loc[proteins_updated, 'Localization']
LD_marker_new_path = Path(label_path) / 'markers_modified_1.txt'
LD_marker_new.to_csv(LD_marker_new_path, sep='\t', index=True)

print(f"Updated localizations for {len(proteins_updated)} proteins.")
print(f"Revised marker file written to {LD_marker_new_path}.")

if len(proteins_updated):
    change_summary = (
        pd.DataFrame({
            'Original_Localization': LD_marker.loc[proteins_updated, 'Localization'],
            'Revised_Localization': LD_marker_new.loc[proteins_updated, 'Localization']
        })
        .value_counts()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    display(change_summary)
else:
    print('No localization discrepancies found between LD_marker and LD.')

Updated localizations for 250 proteins.
Revised marker file written to /home/ec2-user/Jul2025/labels/markers_modified_1.txt.


,Original_Localization,Revised_Localization,count
0,Secretory,Mitochondria,66
1,Cytosol,Nucleus,42
2,Nucleus,Secretory,40
3,Cytosol,Secretory,22
4,Secretory,Nucleus,19
5,Secretory,Cytosol,16
6,Nucleus,Cytosol,11
7,Cytosol,Mitochondria,11
8,Nucleus,Mitochondria,8
9,Mitochondria,Cytosol,5
